## Preparation for Silver layer
This section prepares the raw Bronze data to be promoted to the Silver layer by applying data quality checks and basic transformations.

## 1. Load order items data from the bronze table
- Loads the order items data from the bronze layer into a Spark DataFrame for further processing.

## 2. Display the schema of the order items DataFrame
- Prints the schema of the loaded order items DataFrame to understand its structure and data types.

## 3. Check for null values in each column of the order items DF
- Checks for null values in each column and displays the count to identify data quality issues.

## 4. Remove duplicate rows from the order items DataFrame
- Removes duplicate rows to ensure data integrity and prevent double-counting.

## 5. Count and display the number of nulls in each column after deduplication
- Counts and displays the number of nulls in each column after removing duplicates to verify data cleanliness.

## 6. Further processing
- Reads the bronze_categories table from the bronze_ecommerce schema in the Hive metastore.
- Loads the data into a Spark DataFrame.
- Prints the DataFrame schema and displays the column names and data types for inspection.

## 7. Check for null values in each column of the categories DataFrame
- Checks for null values in each column of the categories DataFrame to assess data quality.

## 8. Remove duplicate rows from the categories DataFrame
- Removes duplicate rows from the categories DataFrame to maintain data accuracy.

## 9. Count and display the number of nulls in each column after deduplication (categories)
- Counts and displays the number of nulls in each column after deduplication to confirm data quality.

## 10. Write the cleaned categories DataFrame to the silver layer
- Writes the cleaned categories DataFrame to the silver layer in Delta format for downstream analytics.

## 11. Customers Dimensional Table
- Loads the bronze_customers table from the bronze_ecommerce schema and prints its schema to prepare for further processing.

## 1. Load order items data from the bronze table
- This cell loads the order items data from the bronze layer into a Spark DataFrame.

In [0]:
df_order_items = spark.read.table("hive_metastore.bronze_ecommerce.bronze_order_items")



## 2. Display the schema of the order items DataFrame
- This cell prints the schema of the loaded order items DataFrame.

In [0]:
df_order_items.printSchema()

## 3. Check for null values in each column of the order items DF
- This cell checks for null values in each column and displays the count.

In [0]:
from pyspark.sql.functions import col, sum as _sum

df_order_items.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_order_items.columns]).display()


## 4. Remove duplicate rows from the order items DataFrame
- This cell removes duplicate rows from the order items DataFrame.

In [0]:
df_order_items = df_order_items.dropDuplicates()


## 5. Count and display the number of nulls in each column after deduplication
- This cell counts and displays the number of nulls in each column after removing duplicates.

In [0]:
from pyspark.sql.functions import col, sum, when

# Count nulls in each column
df_nulls1 = df_order_items.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_order_items.columns])

# Show the results
df_nulls1.display()


## 6. Further processing

- This cell reads the bronze_categories table from the bronze_ecommerce schema in the Hive metastore
- Loads the data into a Spark DataFrame
- Prints the DataFrame schema
- Displays the column names and data types

In [0]:
df_categories = spark.read.table("hive_metastore.bronze_ecommerce.bronze_categories")

df_categories.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as _sum

df_categories.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_categories.columns])

In [0]:
df_categories = df_categories.dropDuplicates()


In [0]:
from pyspark.sql.functions import col, sum, when

# Count nulls in each column
df_nulls2 = df_categories.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_categories.columns])

In [0]:
df_categories.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce.silver_categories")

## Customers Dimensional Table

In [0]:
df_customers = spark.read.table("hive_metastore.bronze_ecommerce.bronze_customers")
df_customers.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as _sum
df_customers.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_customers.columns]).display()

In [0]:
df_customers = df_customers.dropDuplicates()

In [0]:
from pyspark.sql.functions import col, sum, when

# Count nulls in each column
df_nulls3 = df_customers.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_customers.columns])

# Show the results
df_nulls3.display()

In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce.silver_customers")

## Orders Dimensional Table

In [0]:
df_orders = spark.read.table("hive_metastore.bronze_ecommerce.bronze_orders")
df_orders.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as _sum
df_orders.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_orders.columns]).display()

In [0]:
df_orders = df_orders.dropDuplicates()

In [0]:
from pyspark.sql.functions import col, sum, when

# Count nulls in each column
df_nulls4 = df_orders.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_orders.columns])

# Show the results
df_nulls4.display()

In [0]:
df_orders.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce.silver_orders")

## Products Dimensional Table

In [0]:
df_products = spark.read.table("hive_metastore.bronze_ecommerce.bronze_products")
df_products.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as _sum
df_products.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_products.columns]).display()

In [0]:
df_no_nulls = df_clean.dropna()

display(df_no_nulls)


In [0]:
df_no_dupes = df_no_nulls.dropDuplicates()



In [0]:
df_no_dupes.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce.silver_products")

Reviews Dimensional Table

In [0]:
df_reviews = spark.read.table("hive_metastore.bronze_ecommerce.bronze_reviews")
df_reviews.printSchema()

In [0]:
df_clean2.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce.silver_reviews")